### RAG first step: In-memory search

In [ ]:
# The notebook will need couple module files from code directory.
import os,sys
# os.path.join('..', 'code') is the relative path for code folder, module_path convert it to absolute path, final absolute url: d:\\Study\\Python\\llm-zoomcamp\\code
module_path = os.path.abspath(os.path.join('..', 'code'))

In [4]:
# append the code folder in sys.path if not already exists (100% likely)
if module_path not in sys.path:
    sys.path.append(module_path)

In [5]:
from ingest import load_faq_data, build_index

# load knowledge base (json format) from https://datatalks.club/faq/json/courses.json
documents = load_faq_data()
# documents: list of 1208 json object, 24 secs to load

# build_index is using minsearch.Index to build text index probably via TD_IDF
index = build_index(documents)
# index: minsearch.Index object

In [ ]:
# open_api_key need to retrive from .env file
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from rag_helper import RAGBase

# initiate RAGBase with knowledge base index and llm_client
# the remaining args are pre-defined in modular file or have default values.
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [ ]:
# rag function only need to take query. It will search index to find relavent knowledge base and build prompt altogether and send prompt to llm_client
answer = assistant.rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [ ]:
def llm(prompt):
    #response = openai_client.responses.create(
    response = openai_client.chat.completions.create(
        model='gpt-5.4-mini',
        #input=prompt
        messages=[
            {"role":"system","content": "You are a helpful assistent."},
            {"role":"user","content": prompt},
        ]
    )
    #return response.output_text
    return [ x.message.content for x in response.choices]

In [34]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [50]:
response.usage

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00039900000000000005

In [53]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes, you can still join now. If you want a certificate, make sure you submit your project while submissions are still open.


In [54]:
rag('How do I get a certificate?')

'You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.\n\nA few notes:\n- Self-paced mode does not include certificates.\n- Homework is not required for the certificate.\n- You need to peer-review 3 capstone projects, which is only possible while the course is running.\n- Make sure your official name is set in your course profile if you want it to appear correctly on the certificate.'